<a href="https://colab.research.google.com/github/Hanu0745/ai-mentor-portfolio/blob/main/Day_2Resumeextractor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q google-genai pydantic
import os, getpass
if 'GEMINI_API_KEY' not in os.environ:
    os.environ['GEMINI_API_KEY'] = getpass.getpass('Gemini API key: ')

Gemini API key: ··········


In [3]:
from pydantic import BaseModel
from typing import List, Optional

class Education(BaseModel):
    degree: str
    institution: str
    year: int

class Resume(BaseModel):
    name: str
    email: str
    phone: Optional[str] = None
    education: List[Education]
    skills: List[str]
    projects: List[str] = []
    experience_years: float

In [4]:
from google import genai
from pydantic import ValidationError

client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

def extract_resume(raw_text: str, max_retries: int = 1) -> Resume:
    for attempt in range(max_retries + 1):
        try:
            resp = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=f'Extract a Resume JSON from this text. Return ONLY JSON, no markdown.\n\n{raw_text}',
                config={'response_mime_type': 'application/json',
                        'response_schema': Resume.model_json_schema()},
            )
            return Resume.model_validate_json(resp.text)
        except ValidationError as e:
            if attempt == max_retries:
                raise
            # Retry once: ask Gemini to fix the JSON
            fix_prompt = f'Fix this JSON to match schema. Errors: {e}. Original: {resp.text}'
            resp = client.models.generate_content(
                model='gemini-2.5-flash', contents=fix_prompt,
                config={'response_mime_type': 'application/json',
                        'response_schema': Resume.model_json_schema()},
            )
            return Resume.model_validate_json(resp.text)

In [6]:
# Load sample résumés
with open('/content/sample_resumes.txt') as f:
    resumes = [r.strip() for r in f.read().split('---') if r.strip()]

results = []
for i, r in enumerate(resumes[:3]):
    try:
        parsed = extract_resume(r)
        results.append(parsed)
        print(f'Résumé {i+1}: {parsed.name} — {len(parsed.skills)} skills')
    except Exception as e:
        print(f'Résumé {i+1}: FAILED — {e}')

print('\n', results[0].model_dump_json(indent=2) if results else 'no results')

Résumé 1: Ravi Kumar — 6 skills
Résumé 2: Sneha Reddy — 6 skills
Résumé 3: Arun Pillai — 8 skills

 {
  "name": "Ravi Kumar",
  "email": "ravi.kumar@example.com",
  "phone": "+91-9876543210",
  "education": [
    {
      "degree": "B.Tech Computer Science",
      "institution": "Aditya University",
      "year": 2026
    },
    {
      "degree": "Intermediate",
      "institution": "Narayana Junior College",
      "year": 2022
    }
  ],
  "skills": [
    "Python",
    "Java",
    "SQL",
    "Git",
    "Linux",
    "REST APIs"
  ],
  "projects": [
    "Built a Flask REST API for college placement portal (3-week internship at startup)",
    "Solved 250+ DSA problems on LeetCode (Top 5% in college)",
    "Final-year project: ML model for crop yield prediction (Random Forest, 87% accuracy)"
  ],
  "experience_years": 0.25
}


## Pushing to an Existing Git Repository

To push your Colab notebook and related files to an existing Git repository, you'll need to perform the following steps:

1.  **Initialize Git** in your Colab environment (if not already done).
2.  **Configure your Git username and email**.
3.  **Add the remote repository URL**.
4.  **Add your files** to the staging area.
5.  **Commit your changes**.
6.  **Push your committed changes** to the remote repository.

In [7]:
# 1. Initialize Git (if not already initialized)
# This command initializes a new Git repository in the current directory.
# If your Colab environment already has a .git folder, you can skip this.
!git init

hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /content/.git/


In [8]:
# 2. Configure Git User Name and Email
# Replace 'Your Name' and 'your.email@example.com' with your actual Git credentials.
!git config --global user.name "Hanu0745"
!git config --global user.email "bhanumanthu450@gmail.com"

### Important: Authenticating with Git

If your repository is private or requires authentication, you'll typically need to provide credentials. For GitHub, a Personal Access Token (PAT) is the recommended method. You can include your PAT in the remote URL (e.g., `https://<YOUR_PAT>@github.com/username/repo.git`) or you'll be prompted for credentials when you push.

**Note**: Directly embedding sensitive tokens in your notebook is generally not recommended for security reasons. Consider using `getpass` or Colab secrets for better security practices if sharing the notebook.

In [9]:
# 3. Add the Remote Repository URL
# Replace 'your_repo_url' with the actual URL of your existing GitHub repository.
# For example: 'https://github.com/your_username/your_repository.git'

# IMPORTANT: Replace <YOUR_GITHUB_PERSONAL_ACCESS_TOKEN> with your actual PAT.
# Create one here: https://github.com/settings/tokens

# If you've already added a remote named 'origin', you might want to remove it first
# !git remote remove origin

remote_url = "https://Hanu0745:<YOUR_GITHUB_PERSONAL_ACCESS_TOKEN>@github.com/Hanu0745/ai-mentor-portfolio.git"
!git remote add origin {remote_url}

### Adding files to Git

To add all files in your current working directory to the Git staging area, use `git add .`.

**Important**: Colab notebooks (`.ipynb` files) can be quite large due to output data. It's often better to save only the code (clearing outputs) or to explicitly specify which files to add if you don't want to include everything.

In [10]:
# 4. Add files to the staging area
# This command stages all changes in the current directory.
# You might want to be more selective, e.g., '!git add your_notebook_name.ipynb'
!git add .

In [11]:
# 5. Commit your changes
# This command records the staged changes to the repository with a descriptive message.
!git commit -m "Add initial notebook content from Colab"

[master (root-commit) 9475bb1] Add initial notebook content from Colab
 22 files changed, 51143 insertions(+)
 create mode 100644 .config/.last_opt_in_prompt.yaml
 create mode 100644 .config/.last_survey_prompt.yaml
 create mode 100644 .config/.last_update_check.json
 create mode 100644 .config/active_config
 create mode 100644 .config/config_sentinel
 create mode 100644 .config/configurations/config_default
 create mode 100644 .config/default_configs.db
 create mode 100644 .config/gce
 create mode 100644 .config/hidden_gcloud_config_universe_descriptor_data_cache_configs.db
 create mode 100644 .config/logs/2026.05.18/13.25.54.745593.log
 create mode 100644 .config/logs/2026.05.18/13.26.13.953937.log
 create mode 100644 .config/logs/2026.05.18/13.26.29.642900.log
 create mode 100644 .config/logs/2026.05.18/13.26.31.618620.log
 create mode 100644 .config/logs/2026.05.18/13.26.48.703307.log
 create mode 100644 .config/logs/2026.05.18/13.26.49.559528.log
 create mode 100755 sample_data/RE

In [14]:
# 6. Push to the Remote Repository
# This command uploads your local commits to the remote repository's 'main' branch.
# If your default branch is different (e.g., 'master'), replace 'main' accordingly.
!git push -u origin master

fatal: could not read Username for 'https://github.com': No such device or address
